# Part 2 — LoRA 기본 구조 실습 (EXAONE 4.0 1.2B)
### 어댑터만 학습하는 원리를 직접 확인

> **환경**: 윈도우 로컬 VSCode + venv (`Python 3.11 (finetuning)` 커널)  
> **선행**: Part 1 (양자화) 완료  
> **최종 목표**: EXAONE 4.0 1.2B 법률 Q&A 파인튜닝 (Part 6까지)  
> **이번 파트 목표**: LoRA의 4가지 핵심을 **작은 모델로 리허설** → Part 3에서 EXAONE 4.0에 그대로 적용

---

## 🎯 학습 목표

| # | 개념 | 결과물 |
|---|------|--------|
| 1 | 파라미터 동결 (`requires_grad`) | 원본 모델을 얼리는 원리 |
| 2 | 저랭크 분해 `ΔW = A × B` 직접 구현 | 수학적 실체 확인 |
| 3 | r=16 (EXAONE 4.0 1.2B 설정) 파라미터 수 계산 | 1.6% 비율 체득 |
| 4 | `LoraConfig` + `get_peft_model` | 실전 코드 패턴 |
| 5 | 어댑터 저장·로드 | 15~30MB 파일의 정체 |
| 6 | Part 3에서 EXAONE 4.0에 적용할 설정 확정 | `lora_config` 완성 |

**실행 환경 확인:**  
우측 상단 커널이 **`Python 3.11 (finetuning)`** 인지 확인하세요.

---
## 0️⃣ 환경 준비

**이론 요약**
- PEFT 라이브러리(`peft`)가 LoRA 핵심 도구
- 실습용으로 **초소형 모델**(`tiny-random-gpt2`, ~5MB) 사용
- 작은 모델로 원리 확인 후 **Part 3에서 EXAONE 4.0 적용**

### 미션 0-1. 환경 검증

In [ ]:
import sys
import torch
import peft

print(f"Python: {sys.executable}")
print(f"PyTorch: {torch.__version__}")
print(f"peft: {peft.__version__}")

✅ **결과 확인**: Python 경로에 **`venv`** 포함되고, peft 0.14+ 나오면 OK.  
👉 `AppData` 경로면 커널을 `Python 3.11 (finetuning)`로 변경.

---
## 1️⃣ 파라미터 동결 — LoRA의 출발점

**이론 요약**
- LoRA = 원본 모델 **동결** + 어댑터만 학습
- PyTorch에서 동결 = `requires_grad = False` (단 한 줄)
- EXAONE 4.0 12억 개 전체에 이 플래그를 적용한 것이 LoRA의 1단계

```
param.requires_grad = True   → 학습 대상 (기울기 계산)
param.requires_grad = False  → 동결 (기울기 계산 안 함)
```

💡 **왜 중요한가?**  
"1.4% 학습"이라는 말은 이 `requires_grad` 플래그의 결과입니다.

### 미션 1-1. 기본 상태 확인

In [ ]:
import torch.nn as nn

# 간단한 Linear 레이어 1개
layer = nn.Linear(10, 10)

for name, param in layer.named_parameters():
    print(f"{name}: requires_grad={param.requires_grad}")

✅ **결과 확인**: 둘 다 `True`가 기본값.  
👉 PyTorch는 모든 파라미터를 **학습 대상**으로 기본 설정.

### 미션 1-2. 전체 동결

In [ ]:
for param in layer.parameters():
    param.requires_grad = False

for name, param in layer.named_parameters():
    print(f"{name}: requires_grad={param.requires_grad}")

✅ **결과 확인**: 전부 `False`로 바뀌었나요?  
👉 LoRA 내부도 이것과 똑같은 방식으로 12억 개 파라미터를 얼립니다.

---
## 2️⃣ 저랭크 분해 — `ΔW = A × B`

**이론 요약**
- EXAONE 4.0 hidden_size = 2048
- 원본 크기: `2048 × 2048 = 4,194,304개`
- LoRA(r=16): `A(2048×16) + B(16×2048) = 65,536개` = **원본의 1.6%**
- 곱(A×B)은 원본과 **같은 크기** 표현 가능 → "표현력 vs 학습량"의 절묘한 균형

💡 **왜 중요한가?**  
shape을 직접 찍어보면 "어떻게 1.4%만 학습이 가능한가"가 즉시 이해됩니다.

### 미션 2-1. 원본 ΔW 만들기 (기준)

In [ ]:
차원 = 2048  # EXAONE 4.0 1.2B 한 레이어 크기 (hidden_size)

원본_ΔW = torch.randn(차원, 차원)
print(f"원본 ΔW shape: {원본_ΔW.shape}")
print(f"파라미터 수: {원본_ΔW.numel():,}개")

### 미션 2-2. LoRA 분해 (r=16, EXAONE 4.0 1.2B 설정)

In [ ]:
r = 16  # EXAONE 4.0 1.2B의 lora_r (3.5의 32 → 16으로 줄임)

A = torch.randn(차원, r)
B = torch.randn(r, 차원)

print(f"A shape: {A.shape}, 파라미터: {A.numel():,}개")
print(f"B shape: {B.shape}, 파라미터: {B.numel():,}개")
print(f"합계: {A.numel() + B.numel():,}개")

### 미션 2-3. A × B 결과 확인

In [ ]:
ΔW_LoRA = A @ B

print(f"A × B shape: {ΔW_LoRA.shape}")
print(f"원본과 같은가? {ΔW_LoRA.shape == 원본_ΔW.shape}")

✅ **결과 확인**: `(2048, 2048)`으로 원본과 같은 크기인가요?  
👉 **같은 크기의 변화량을 1.6% 파라미터로 표현** — 이것이 LoRA의 마법.

### 미션 2-4. 절감률 계산

In [ ]:
원본_개수 = 원본_ΔW.numel()
LoRA_개수 = A.numel() + B.numel()

절감률 = (1 - LoRA_개수 / 원본_개수) * 100

print(f"원본:  {원본_개수:>12,}개 (100.00%)")
print(f"LoRA: {LoRA_개수:>12,}개 ({LoRA_개수/원본_개수*100:.2f}%)")
print(f"절감률: {절감률:.2f}%")

✅ **결과 확인**: 98% 이상 절감되나요?  
👉 r=16이어도 저장·학습 비용은 원본의 1.6% 수준.

---
## 3️⃣ rank(r) 선택 가이드

**이론 요약**
- r이 클수록: 학습 파라미터 ↑, 표현력 ↑, 메모리 ↑
- **EXAONE 4.0 1.2B 선택**: `r=16` (모델 크기에 적절)
- EXAONE 3.5 Full 버전(2.4B)은 r=32 — 더 큰 모델이라 더 큰 r

💡 **왜 중요한가?**  
이번 과정에서 **r=16을 쓰는 근거**를 숫자로 확인.

### 미션 3-1. r별 파라미터 수 비교표

In [ ]:
차원 = 2048  # EXAONE 4.0
원본_개수 = 차원 * 차원

print(f"{'r':>5} {'A+B 파라미터':>15} {'원본 대비':>10} {'용도':>40}")
print("-" * 75)

r_가이드 = {
    4: "단순 분류",
    8: "일반 Q&A (기본값)",
    16: "EXAONE 4.0 1.2B 법률 Q&A ← 이번 과정 ✨",
    32: "EXAONE 3.5 2.4B (Full 버전)",
    64: "코드 생성",
}
for r, 용도 in r_가이드.items():
    LoRA_개수 = 2 * 차원 * r
    비율 = LoRA_개수 / 원본_개수 * 100
    표시 = f"r={r}"
    print(f"{표시:>5} {LoRA_개수:>15,} {비율:>9.2f}% {용도:>40s}")

✅ **결과 확인**: r=16일 때 약 **1.56%** 나왔나요?  
👉 Full 버전(r=32)의 절반 → EXAONE 4.0 1.2B의 작은 크기에 적절.

---
## 4️⃣ alpha — 어댑터 영향력 조절

**이론 요약**
- 실제 변화량 = `ΔW × (alpha / r)`
- 실무 표준: **`alpha = r × 2`**
- EXAONE 4.0 과정: r=16 → **alpha=32**

💡 **왜 중요한가?**  
Full 버전(3.5)의 `lora_alpha=64`가 우리 버전(4.0)에서는 `32`인 근거 체득.

### 미션 4-1. alpha에 따른 스케일 팩터

In [ ]:
r = 16
후보 = [8, 16, 32, 64]

print(f"{'alpha':>6} {'alpha/r':>10} {'설명':>30}")
print("-" * 50)

for alpha in 후보:
    스케일 = alpha / r
    설명 = "기본 (1.0x)" if 스케일 == 1.0 else (
        "2배 ✅ EXAONE 4.0 권장" if 스케일 == 2.0 else f"{스케일}x"
    )
    print(f"{alpha:>6} {스케일:>10.1f} {설명:>30s}")

### 미션 4-2. 실제 스케일링 효과

In [ ]:
alpha = 32  # EXAONE 4.0 과정 설정

A = torch.randn(차원, r) * 0.01   # LoRA 초기화는 작은 값
B = torch.randn(r, 차원) * 0.01

스케일 = alpha / r
ΔW_기본 = A @ B
ΔW_스케일드 = (A @ B) * 스케일

print(f"스케일 적용 전 평균 절댓값: {ΔW_기본.abs().mean().item():.6f}")
print(f"스케일 {스케일}x 적용 후:    {ΔW_스케일드.abs().mean().item():.6f}")
print(f"증가율: {ΔW_스케일드.abs().mean().item() / ΔW_기본.abs().mean().item():.1f}배")

✅ **결과 확인**: 정확히 2배 증가했나요?  
👉 학습 중에 이 스케일링이 **매 스텝 자동 적용**됩니다.

---
## 5️⃣ PEFT로 LoRA 실전 적용

**이론 요약**
- `LoraConfig` → 설정  
- `get_peft_model` → 어댑터 부착  
- `print_trainable_parameters()` → 1.4% 검증

실습용으로 **초소형 GPT-2**(safetensors 포맷) 사용 → 빠르게 원리 확인.

💡 **왜 작은 모델로?**  
EXAONE 4.0은 2.5GB 다운로드라 매번 로드하기 부담. 작은 모델로 **PEFT 패턴을 완전히 익힌 뒤** Part 3에서 EXAONE 4.0 적용.

### 미션 5-1. 테스트 모델 로드

In [ ]:
from transformers import AutoModelForCausalLM

테스트_MODEL = "hf-internal-testing/tiny-random-gpt2"  # ~5MB

model = AutoModelForCausalLM.from_pretrained(테스트_MODEL)
print(f"✅ 테스트 모델 로드 완료")

### 미션 5-2. LoRA 적용 전 파라미터 수

In [ ]:
전체 = sum(p.numel() for p in model.parameters())
학습가능 = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"전체 파라미터:   {전체:,}")
print(f"학습 가능:       {학습가능:,}")
print(f"학습 비율:       {학습가능/전체*100:.2f}%")

✅ **결과 확인**: 전체 == 학습가능 == 100% 인가요?  
👉 LoRA 적용 전에는 **모든 파라미터가 학습 대상**.

### 미션 5-3. LoraConfig 만들기 (EXAONE 4.0 설정값 사용)

In [ ]:
from peft import LoraConfig, TaskType

lora_config = LoraConfig(
    r=16,                                   # EXAONE 4.0 1.2B용
    lora_alpha=32,                          # r × 2
    target_modules=["c_attn"],              # GPT-2 (테스트 모델용)
    lora_dropout=0.05,                      # 과적합 방지
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

print("✅ LoraConfig 생성 완료")
print(f"  r:          {lora_config.r}")
print(f"  lora_alpha: {lora_config.lora_alpha}")
print(f"  스케일:      {lora_config.lora_alpha / lora_config.r}x")

> 💡 **Part 3에서 EXAONE 4.0 적용 시 바뀔 것**:  
> `target_modules=["c_attn"]` → EXAONE 4.0의 Llama 계열 구조:  
> `["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]`  
> 나머지 설정(r=16, alpha=32, dropout=0.05, bias, task_type)은 **완전히 동일**.

### 미션 5-4. 어댑터 부착

In [ ]:
from peft import get_peft_model

lora_model = get_peft_model(model, lora_config)
print("✅ LoRA 어댑터 부착 완료")

### 미션 5-5. 학습 파라미터 수 확인 (핵심!)

In [ ]:
lora_model.print_trainable_parameters()

✅ **결과 확인**: `trainable%`가 표시되나요?  
👉 이 한 줄이 LoRA의 전부를 증명.  
👉 EXAONE 4.0 1.2B에 똑같이 r=16 적용 시 **약 1.4%** (1,650만 / 12억) 예상.

### 미션 5-6. 동결 vs 학습 파라미터 직접 확인

In [ ]:
동결 = sum(p.numel() for p in lora_model.parameters() if not p.requires_grad)
학습 = sum(p.numel() for p in lora_model.parameters() if p.requires_grad)

print(f"동결된 원본:   {동결:>10,}개")
print(f"학습할 어댑터: {학습:>10,}개")
print(f"학습 비율:     {학습/(동결+학습)*100:.3f}%")

✅ **결과 확인**: 원본이 **훨씬 크게** 나왔나요?  
👉 1단계 `requires_grad=False`가 수백만 개에 자동 적용된 결과.

---
## 6️⃣ 어댑터가 실제로 어디에 붙었는지 확인

**이론 요약**
- `target_modules`에 지정한 레이어에만 LoRA 추가
- LoRA 레이어 이름에 `lora_A`, `lora_B` 포함
- 디버깅의 기본

💡 **왜 중요한가?**  
`target_modules` 잘못 지정 시 **에러 또는 학습 안 됨**. 눈으로 확인하는 습관 필수.

### 미션 6-1. LoRA 레이어만 골라 출력

In [ ]:
lora_개수 = 0
for name, module in lora_model.named_modules():
    if "lora" in name.lower():
        lora_개수 += 1

print(f"LoRA 관련 레이어: {lora_개수}개")

# 처음 5개만 예시 출력
print("\n샘플 5개:")
개수 = 0
for name, _ in lora_model.named_modules():
    if "lora" in name.lower() and 개수 < 5:
        print(f"  {name}")
        개수 += 1

✅ **결과 확인**: `lora_A`, `lora_B` 이름이 보이나요?  
👉 이 레이어들이 **실제로 학습되는 부분**.

---
## 7️⃣ 어댑터 저장 & 로드

**이론 요약**
- `save_pretrained(경로)` → 어댑터만 저장 (원본 모델 제외)
- 저장 파일: `adapter_config.json` + `adapter_model.safetensors`
- 원본 모델 + 어댑터 → 재현 가능

💡 **왜 중요한가?**  
Part 6에서 학습된 EXAONE 4.0 어댑터를 **HuggingFace Hub에 공유**합니다. 저장 구조 이해 필수.

### 미션 7-1. 어댑터 저장

In [ ]:
import os

저장경로 = "./test_adapter"
lora_model.save_pretrained(저장경로)
print(f"✅ 저장 완료: {저장경로}")

# 저장된 파일 확인
for 파일 in sorted(os.listdir(저장경로)):
    크기 = os.path.getsize(os.path.join(저장경로, 파일))
    print(f"  {파일}: {크기/1024:.1f} KB")

✅ **결과 확인**: `adapter_config.json`과 `adapter_model.safetensors` 두 개 있나요?  
👉 어댑터의 전부는 이 두 파일.

### 미션 7-2. adapter_config.json 내용 확인

In [ ]:
import json

with open(f"{저장경로}/adapter_config.json", "r") as f:
    설정 = json.load(f)

for 키 in ["r", "lora_alpha", "target_modules", "lora_dropout", "bias", "task_type"]:
    값 = 설정.get(키)
    # target_modules는 set으로 저장될 수 있으므로 list로 변환해 표시
    if isinstance(값, list):
        값 = sorted(값)
    print(f"{키}: {값}")

✅ **결과 확인**: 미션 5-3에서 설정한 값이 그대로 저장됐나요?  
👉 이 파일만 공유하면 **완전히 재현 가능**.

### 미션 7-3. 어댑터 로드 (재사용)

In [ ]:
from peft import PeftModel

# 원본 모델을 새로 로드 (학습 안 된 깨끗한 상태)
원본_재로드 = AutoModelForCausalLM.from_pretrained(테스트_MODEL)

# 어댑터 끼워 넣기
로드된_모델 = PeftModel.from_pretrained(원본_재로드, 저장경로)
print("✅ 어댑터 로드 완료")

로드된_모델.print_trainable_parameters()

✅ **결과 확인**: 미션 5-5에서 본 숫자와 **동일**하게 나왔나요?  
👉 어댑터가 정확히 복원됐다는 증거.

---
## 8️⃣ Part 3에서 사용할 설정 확정

**이론 요약**
- Part 2에서 검증한 패턴을 Part 3에서 EXAONE 4.0에 그대로 적용
- `target_modules`만 EXAONE 4.0 Llama 계열 구조로 교체
- Part 1의 `bnb_config`와 결합하면 **QLoRA 완성**

### 미션 8-1. EXAONE 4.0용 최종 LoraConfig 확정

In [ ]:
# Part 3에서 EXAONE 4.0 1.2B에 적용할 설정
EXAONE4_lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",      # Attention 4개 tttttt
        "gate_proj", "up_proj", "down_proj",         # MLP 3개 gggggggg(SwiGLU)
    ],                                               # 👈 EXAONE 4.0용 ttttttttt8888888(Llama 계열)
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

print("📋 Part 3에서 쓸 LoraConfig (EXAONE 4.0 1.2B):")
print(f"  r:               {EXAONE4_lora_config.r}")
print(f"  lora_alpha:      {EXAONE4_lora_config.lora_alpha}")
print(f"  target_modules:  {sorted(EXAONE4_lora_config.target_modules) if isinstance(EXAONE4_lora_config.target_modules, (list, set)) else EXAONE4_lora_config.target_modules}")
print(f"  lora_dropout:    {EXAONE4_lora_config.lora_dropout}")
print(f"  task_type:       {EXAONE4_lora_config.task_type}")
print(f"  bias:            {EXAONE4_lora_config.bias}")

> 💡 Part 3에서 이 설정을 **그대로 복사**해서 쓰면 됩니다.  
> EXAONE 4.0은 HF 공식 지원이라 3.5의 호환성 패치가 **전혀 필요 없습니다**!

---

## 🎓 Part 2 정리

| 미션 | 배운 것 | EXAONE 4.0과의 연결 |
|---|---|---|
| 1 | `requires_grad=False` | EXAONE 4.0 12억 개 동결 원리 |
| 2 | `ΔW = A × B` 저랭크 분해 | LoRA 수학적 실체 |
| 3 | r=16의 근거 (1.6%) | EXAONE 4.0 1.2B에 적절한 크기 |
| 4 | alpha=32 (r×2)의 근거 | 실무 표준 |
| 5 | `LoraConfig` + `get_peft_model` | Part 3 실전 코드 패턴 |
| 6 | 어댑터 위치 확인 | 디버깅 방법 |
| 7 | 어댑터 저장·로드 | Part 6 HF 업로드 기초 |
| 8 | EXAONE 4.0용 설정 확정 | Part 3 준비 완료 |

### 🔑 EXAONE 4.0 핵심 포인트

| 항목 | EXAONE 3.5 (Full 버전) | **EXAONE 4.0 (이번)** |
|---|---|---|
| r | 32 | **16** (모델 작으니까) |
| alpha | 64 | **32** (r×2 유지) |
| Attention | `q,k,v,out_proj` | **`q,k,v,o_proj`** |
| MLP | `c_fc_0,c_fc_1,c_proj` | **`gate_proj,up_proj,down_proj`** |
| 호환성 패치 | 필요 | **불필요 ✅** |

### 🔜 다음 파트

**Part 3 — QLoRA 통합**

```python
# Part 3에서 만들 코드 (미리보기)

# 1. Part 1의 bnb_config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# 2. EXAONE 4.0 4-bit 로드 (trust_remote_code 불필요!)
model = AutoModelForCausalLM.from_pretrained(
    "LGAI-EXAONE/EXAONE-4.0-1.2B",
    quantization_config=bnb_config,
    device_map="auto",
)

# 3. 학습 가능하게 준비
model = prepare_model_for_kbit_training(model)

# 4. Part 2의 lora_config 적용
model = get_peft_model(model, EXAONE4_lora_config)  # ← Part 2에서 확정

# 5. 학습 파라미터 확인
model.print_trainable_parameters()
# 예상: trainable params: ~16M || all params: ~1.2B || trainable%: ~1.4%
```

**QLoRA** = Part 1(Q) + Part 2(LoRA) = **Part 3**  
**EXAONE 4.0** = HF 공식 지원으로 3.5 패치 불필요 ✨